In [ ]:
import supermarq as sm
import qiskit
import qiskit_superstaq as qss
from qiskit import Aer, execute, QuantumCircuit


from qiskit.converters import circuit_to_dag
from qiskit.tools.visualization import dag_drawer
# dag = circuit_to_dag(circ)
# dag_drawer(dag)

from pprint import pprint
import sys
import numpy
numpy.set_printoptions(threshold=sys.maxsize)

In [ ]:
import termcolor
def matrix_print(nparray, size=8):
    for x in nparray:
        for y in x:
            r = '{:+02.1f}'.format(y.real)
            i = '{:+02.1f}'.format(y.imag)
            if i not in ['+0.0', '-0.0'] or r not in ['+0.0', '-0.0']:
                r = termcolor.colored(r, 'red')
                i = termcolor.colored(i, 'red')
            print("(", r, ",", i, ")",sep="", end="  ")
        print(end="\n\n")
def remove_gates(circ, to_remove=['measure', 'reset', 'h']):
    marked = []
    for ins in circ.data:
        if ins[0].name in to_remove:
            marked.append(ins)
    marked = marked[::-1]
    for each in marked:
        circ.data.remove(each)
    return circ
def get_circ(cir_name, n_qubits):
    if cir_name == 'ghz':
        special = sm.benchmarks.ghz.GHZ(num_qubits=n_qubits)

    elif cir_name == 'ham':
        special = sm.benchmarks.hamiltonian_simulation.HamiltonianSimulation(n_qubits, 1, 1)

    elif cir_name == 'mermin_bell':
        special = sm.benchmarks.mermin_bell.MerminBell(n_qubits)

    special_circ = sm.converters.cirq_to_qiskit(special.circuit())
    
    # removing measurements to run on unitary simulator
    special_circ.remove_final_measurements(inplace=True)
    
    ## seperate initial and final Hadamards: TODO
    special_circ = remove_gates(special_circ)
    
    return special_circ

from qiskit.extensions.unitary import UnitaryGate
from numpy import savez_compressed
def write_gates_to_file(circ, n, directory):
    ins_num = 1
    for ins in circ.data:
        print(ins)
        qc = QuantumCircuit(n)
        qc.data = [ins]
        unitary_backend = Aer.get_backend('unitary_simulator')
        job = execute(qc, unitary_backend, shots=10)
        mat = job.result().get_unitary(qc, n).data
        matrix_print(mat)
        # gate = UnitaryGate(ins.operation)
        # mat = gate.to_matrix()
        # matrix_print(mat)
        savez_compressed(directory + '/gate_' + str(ins_num) + '.npz', mat)
        ins_num += 1

In [ ]:
import os
for c in ['ghz', 'ham', 'mermin_bell']:
    for q in [3,4,5,6,7,8,9,10,11,12]:
        special_circ = get_circ(c, q)
        d = './benchmarks/' + c + '/' + str(q)
        if not os.path.exists('./benchmarks/' + c):
            os.mkdir('./benchmarks/' + c)
        if not os.path.exists(d):
            os.mkdir(d)
        write_gates_to_file(special_circ, q, d)